# /geotag — Endpoint Evaluation

For each fixture article shows:
- **Input text** and headline
- **Expected** city / scope / place count / street count
- **Detected** places (with type + coords), cities (with confidence), streets (with spans), scope
- Per-check pass/fail and overall result

**Prerequisite:** NLP service running. `/readyz` → 200.

In [14]:
import sys, time, requests
sys.path.insert(0, '.')
from _scorecard import load_fixture, print_scorecard, NLP_BASE_URL, HEADERS

cases = load_fixture('geotag_cases.json')
print(f'Loaded {len(cases)} test cases')

Loaded 10 test cases


In [15]:
r = requests.get(f'{NLP_BASE_URL}/readyz', headers=HEADERS)
assert r.status_code == 200, f'Service not ready: {r.status_code} {r.text}'
print('Service ready')

Service ready


In [16]:
def chk(ok: bool) -> str:
    return '✓' if ok else '✗'


def print_geotag_detail(case: dict, data: dict, latency: float) -> None:
    # ── Input ────────────────────────────────────────────────────────────────
    print(f'  HEADLINE : {case.get("headline", "")}')
    print(f'  TEXT     : {case["text"]}')
    print()

    # ── Expected ─────────────────────────────────────────────────────────────
    exp_city        = case.get('expected_city')
    exp_present     = case.get('expected_city_present', True)
    exp_scope       = case.get('expected_geo_scope')
    exp_places_min  = case.get('expected_places_min', 0)
    exp_streets_min = case.get('expected_streets_min', 0)

    print('  EXPECTED:')
    city_label = repr(exp_city) if exp_present else 'none (national/regional)'
    print(f'    city          : {city_label}')
    print(f'    geo_scope     : {exp_scope!r}')
    print(f'    places (min)  : {exp_places_min}')
    print(f'    streets (min) : {exp_streets_min}')
    if case.get('notes'):
        print(f'    note          : {case["notes"]}')
    print()

    # ── Detected places ──────────────────────────────────────────────────────
    all_places  = data.get('all_places', [])
    geo_cities  = data.get('geo_cities', [])
    geo_streets = data.get('geo_streets', [])
    geo_points  = data.get('geo_points', [])

    print(f'  DETECTED PLACES ({len(all_places)}):')
    if all_places:
        for p in all_places:
            coords = f'  [{p["lat"]:.4f}, {p["lon"]:.4f}]' if p.get('lat') else '  [no coords]'
            print(f'    [{p["type"]:8s}]  {p["text"]!r:40s}{coords}')
    else:
        print('    (none)')
    print()

    print(f'  DETECTED CITIES ({len(geo_cities)}):')
    if geo_cities:
        for c in geo_cities:
            print(f'    {c["city_name"]!r:25s}  confidence={c["confidence"]:.3f}')
    else:
        print('    (none)')
    print()

    print(f'  DETECTED STREETS ({len(geo_streets)}):')
    if geo_streets:
        for s in geo_streets:
            edge_info = f'{len(s["edge_ids"])} edge(s)' if s.get('edge_ids') else 'unresolved'
            print(f'    {s["span"]!r:40s}  {edge_info}')
    else:
        print('    (none)')
    print()

    if geo_points:
        print(f'  DETECTED GEO POINTS ({len(geo_points)}):')
        for p in geo_points:
            print(f'    {p["span"]!r:40s}  [{p["lat"]:.4f}, {p["lon"]:.4f}]')
        print()

    # ── Check results ────────────────────────────────────────────────────────
    top_city = geo_cities[0]['city_name'] if geo_cities else None
    if exp_present:
        city_ok = top_city == exp_city
    else:
        city_ok = top_city is None
    scope_ok    = data['geo_scope'] == exp_scope
    places_ok   = len(all_places) >= exp_places_min
    streets_ok  = len(geo_streets) >= exp_streets_min

    print('  CHECKS:')
    print(f'    {chk(city_ok)}  city     : got {top_city!r}  expected {exp_city!r}')
    print(f'    {chk(scope_ok)}  scope    : got {data["geo_scope"]!r}  expected {exp_scope!r}')
    print(f'    {chk(places_ok)}  places   : got {len(all_places)}  min {exp_places_min}')
    print(f'    {chk(streets_ok)}  streets  : got {len(geo_streets)}  min {exp_streets_min}')
    print(f'    latency : {latency:.2f}s')

    return city_ok, scope_ok, places_ok, streets_ok

In [17]:
results = []

for case in cases:
    t0 = time.monotonic()
    resp = requests.post(
        f'{NLP_BASE_URL}/geotag',
        json={
            'article_id': case['article_id'],
            'text':       case['text'],
            'headline':   case.get('headline', ''),
            'source':     case.get('source', ''),
        },
        headers=HEADERS,
    )
    latency = time.monotonic() - t0
    assert resp.status_code == 200, \
        f"{case['article_id']}: HTTP {resp.status_code} — {resp.text}"
    data = resp.json()

    city_ok, scope_ok, places_ok, streets_ok = print_geotag_detail(case, data, latency)
    passed = city_ok and scope_ok and places_ok and streets_ok
    icon   = '✅' if passed else '❌'

    print(f'  {icon} [{case["article_id"]}]  {case["description"]}')
    print()
    print('═' * 72)
    print()

    results.append({
        'id':         case['article_id'],
        'city_ok':    city_ok,
        'scope_ok':   scope_ok,
        'places_ok':  places_ok,
        'streets_ok': streets_ok,
        'latency_s':  latency,
        'pass':       passed,
    })

  HEADLINE : Nuevo carril bici en Gran Vía de Madrid
  TEXT     : El Ayuntamiento de Madrid ha inaugurado un nuevo tramo de carril bici en la Gran Vía que conecta con la red existente.

  EXPECTED:
    city          : 'Madrid'
    geo_scope     : 'city'
    places (min)  : 1
    streets (min) : 0

  DETECTED PLACES (2):
    [other   ]  'Gran Vía de Madrid'                      [no coords]
    [other   ]  'Gran Vía'                                [no coords]

  DETECTED CITIES (0):
    (none)

  DETECTED STREETS (0):
    (none)

  CHECKS:
    ✗  city     : got None  expected 'Madrid'
    ✓  scope    : got 'city'  expected 'city'
    ✓  places   : got 2  min 1
    ✓  streets  : got 0  min 0
    latency : 0.60s
  ❌ [geo-001]  Single unambiguous city mention

════════════════════════════════════════════════════════════════════════

  HEADLINE : Valladolid invierte en infraestructura ciclista
  TEXT     : El Ayuntamiento de Valladolid aprobó ayer un plan de 3 millones de euros para ampliar 

In [18]:
passing    = [r for r in results if r['pass']]
city_pass  = sum(1 for r in results if r['city_ok'])
scope_pass = sum(1 for r in results if r['scope_ok'])
avg_lat    = sum(r['latency_s'] for r in results) / len(results)

print_scorecard('/geotag', {
    'Cases':                   len(results),
    'Passing (all checks)':    f'{len(passing)}/{len(results)}',
    'City detection accuracy': f'{city_pass}/{len(results)}',
    'Scope accuracy':          f'{scope_pass}/{len(results)}',
    'Avg latency (s)':         avg_lat,
})


  /geotag
  Cases                               10
  Passing (all checks)                2/10
  City detection accuracy             3/10
  Scope accuracy                      7/10
  Avg latency (s)                     0.320



In [ ]:

# ── Scope passthrough check ───────────────────────────────────────────────────
# For each case, pass the geotag geo_scope into /classify and verify the
# classifier uses it (scope NLI is skipped; returned scope must match exactly).
print('SCOPE PASSTHROUGH CHECK')
print('Calls /classify with geo_scope from /geotag — scope NLI is skipped.')
print()

scope_results = []

for case, result in zip(cases, results):
    geo_scope = None
    # Re-call /geotag to get the geo_scope (results dict only stored pass/fail)
    resp_geo = requests.post(
        f'{NLP_BASE_URL}/geotag',
        json={
            'article_id': case['article_id'],
            'text':       case['text'],
            'headline':   case.get('headline', ''),
            'source':     case.get('source', ''),
        },
        headers=HEADERS,
    )
    if resp_geo.status_code == 200:
        geo_scope = resp_geo.json()['geo_scope']

    # Call /classify passing geo_scope through
    resp_cls = requests.post(
        f'{NLP_BASE_URL}/classify',
        json={
            'article_id':  case['article_id'],
            'summary':     case['text'],   # article text as proxy summary
            'geo_cities':  [],
            'search_tags': [],
            'geo_scope':   geo_scope,
        },
        headers=HEADERS,
    )
    if resp_cls.status_code != 200:
        print(f'  ❌ [{case["article_id"]}]  /classify HTTP {resp_cls.status_code}')
        continue

    cls_data    = resp_cls.json()
    cls_scope   = cls_data['geo_scope']
    scope_match = cls_scope == geo_scope
    icon        = '✓' if scope_match else '✗'
    oos         = cls_data['out_of_scope']

    scope_results.append(scope_match)
    print(f'  {icon} [{case["article_id"]}]  '
          f'geotag={geo_scope!r}  →  classify={cls_scope!r}  '
          f'out_of_scope={oos}  topics={cls_data["topics"]}')

passthrough_ok = sum(scope_results)
print()
print(f'Scope passthrough: {passthrough_ok}/{len(scope_results)} match')
